In [0]:
# COMMAND ----------

from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

MODEL_TABLE = "sandbox.z_jungryo_lee.wls_model_summary_with_corr"
COEF_TABLE = "sandbox.z_jungryo_lee.wls_coef_detail_with_corr"
NETWORK_EDGE_TABLE = "sandbox.z_jungryo_lee.wls_network_edges_with_corr"

CORR_THRESHOLD = 0.1
NETWORK_PVALUE_THRESHOLD = 0.05
NETWORK_MIN_OBS = 10
NETWORK_EDGE_WEIGHT_THRESHOLD = 0.0
DRIVER_PVALUE_THRESHOLD = 0.1
DRIVER_COEF_THRESHOLD = 0.1
NETWORK_EDGE_EXPORT_COLUMNS = [
    "group_dim", "group_key", "y_feature", "x_feature",
    "coef_sign", "corr_abs", "edge_weight",
    "r_squared", "y_obs", "is_significant", "data_created_dt"
]

# 1) 테이블 로드
df_meta = spark.table("sandbox.z_jungryo_lee.buzz_model_meta").toPandas()
df_voc = spark.table("sandbox.z_jungryo_lee.buzz_sc_score_dataset").toPandas()

# 2) JOIN
df_long = df_voc.merge(df_meta, on="model_id", how="left")

# 3) Wide 변환
df_score = (
    df_long
    .pivot_table(index="model_id", columns="category", values="avg_sc")
    .add_suffix("_score")
)

df_count = (
    df_long
    .pivot_table(index="model_id", columns="category", values="total_count")
    .add_suffix("_count")
)

df_wide = pd.concat([df_score, df_count], axis=1).fillna(0)

# COMMAND ----------

def clean_name(s: str) -> str:
    return (
        str(s)
        .replace('.', '')
        .replace('/', '_')
        .replace('(', '')
        .replace(')', '')
        .replace('-', '_')
        .replace(' ', '_')
    )

def clean_col_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [clean_name(c) for c in df.columns]
    return df

def clean_feature_pool(feature_pool: list) -> list:
    return [clean_name(f) for f in feature_pool]


def safe_scalar(value):
    if pd.isna(value):
        return None
    return float(value) if isinstance(value, (np.floating, float, np.integer, int)) else value

df_wide = clean_col_names(df_wide)

# COMMAND ----------

def weighted_corr(x, y, w):
    x = np.asarray(x)
    y = np.asarray(y)
    w = np.asarray(w)

    w_sum = np.sum(w)
    if w_sum == 0:
        return np.nan

    mx = np.sum(w * x) / w_sum
    my = np.sum(w * y) / w_sum

    cov = np.sum(w * (x - mx) * (y - my))
    var_x = np.sum(w * (x - mx) ** 2)
    var_y = np.sum(w * (y - my) ** 2)

    if var_x == 0 or var_y == 0:
        return np.nan

    return cov / np.sqrt(var_x * var_y)

# COMMAND ----------

# DBTITLE 1,Cell 5
def select_x_by_weighted_corr(
    pdf,
    y_feature,
    x_features,
    corr_threshold=CORR_THRESHOLD
):
    y_score_col = f"{y_feature}_score"
    y_count_col = f"{y_feature}_count"

    y = pdf[y_score_col]
    w = np.sqrt(pdf[y_count_col])  # ✅ 권장 1

    selected = []

    for x in x_features:
        if x == y_feature:  # ✅ Y 변수는 X 후보에서 제외
            continue
        x_score_col = f"{x}_score"
        if x_score_col not in pdf.columns:
            continue

        corr = weighted_corr(pdf[x_score_col], y, w)

        if pd.notna(corr) and abs(corr) >= corr_threshold:
            selected.append((x, corr))

    return selected

# COMMAND ----------

def run_wls_weighted(pdf, y_feature, x_features, corr_threshold=CORR_THRESHOLD):

    y_score_col = f"{y_feature}_score"
    y_count_col = f"{y_feature}_count"

    pdf = pdf[
        (pdf[y_count_col] > 0) &
        pdf[y_score_col].notna()
    ]

    if pdf.empty:
        return None, None, None

    # 1) 가중 상관으로 X 필터
    selected_x = select_x_by_weighted_corr(
        pdf, y_feature, x_features, corr_threshold
    )

    if len(selected_x) == 0:
        return None, None, None

    final_x = [x for x, _ in selected_x]

    # 2) WLS
    X = pdf[[f"{x}_score" for x in final_x]]
    X = sm.add_constant(X)
    y = pdf[y_score_col]

    weights = np.sqrt(pdf[y_count_col])  # ✅ 권장 1과 일관

    model = sm.WLS(y, X, weights=weights).fit()

    x_obs_map = {
        x: int((pdf.get(f"{x}_count", 0) > 0).sum())
        for x, _ in selected_x
    }

    return model, selected_x, x_obs_map


def build_network_tables(
    df_model: pd.DataFrame,
    df_coef: pd.DataFrame,
    pvalue_threshold: float = NETWORK_PVALUE_THRESHOLD,
    min_obs: int = NETWORK_MIN_OBS,
    min_edge_weight: float = NETWORK_EDGE_WEIGHT_THRESHOLD
):
    edge_columns = [
        "group_dim", "group_key", "y_feature", "x_feature",
        "coef", "abs_coef", "coef_sign", "p_value", "t_value",
        "x_obs", "driver_rank", "is_driver",
        "weighted_corr", "corr_abs", "edge_weight",
        "r_squared", "y_obs", "is_significant", "data_created_dt"
    ]

    merged = df_coef.merge(
        df_model,
        on=["group_dim", "group_key", "y_feature"],
        how="left"
    )

    if merged.empty:
        return pd.DataFrame(columns=edge_columns)

    merged["abs_coef"] = merged["coef"].abs()
    merged["corr_abs"] = merged["weighted_corr"].abs()
    merged["coef_sign"] = np.where(merged["coef"] >= 0, "positive", "negative")
    merged["is_significant"] = merged["p_value"] < pvalue_threshold
    merged["edge_weight"] = merged["abs_coef"] * merged["corr_abs"]

    df_edges_all = merged[
        merged["y_obs"] >= min_obs
    ].copy()

    if min_edge_weight > 0:
        df_edges_all = df_edges_all[df_edges_all["edge_weight"] >= min_edge_weight].copy()

    return df_edges_all[edge_columns].sort_values(
        ["group_dim", "group_key", "y_feature", "abs_coef"],
        ascending=[True, True, True, False]
    ).reset_index(drop=True)


def plot_ego_network(
    df_edges: pd.DataFrame,
    y_feature: str,
    variable1: str,
    variable1_value: str,
    top_n: int = 8
):
    plot_pdf = df_edges[
        (df_edges["group_dim"] == variable1) &
        (df_edges["group_key"] == variable1_value) &
        (df_edges["y_feature"] == y_feature) &
        df_edges["is_significant"] &
        (df_edges["y_obs"] >= NETWORK_MIN_OBS)
    ].sort_values("abs_coef", ascending=False).head(top_n)

    if plot_pdf.empty:
        print("선택한 조건에서 시각화할 유의한 네트워크 edge가 없습니다.")
        return

    center_x = 0.5
    center_y = 0.5
    radius = 0.38
    angles = np.linspace(0, 2 * np.pi, len(plot_pdf), endpoint=False)
    max_coef = plot_pdf["abs_coef"].max() or 1.0

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(center_x, center_y, s=2200, color="#f6bd60", edgecolor="black")
    ax.text(center_x, center_y, y_feature, ha="center", va="center", fontsize=11)

    for idx, (_, row) in enumerate(plot_pdf.iterrows()):
        node_x = center_x + radius * np.cos(angles[idx])
        node_y = center_y + radius * np.sin(angles[idx])
        color = "#2a9d8f" if row["coef"] >= 0 else "#e76f51"
        line_width = 1.5 + (4.5 * row["abs_coef"] / max_coef)

        ax.scatter(node_x, node_y, s=1400, color="#d9d9d9", edgecolor="black")
        ax.text(node_x, node_y, row["x_feature"], ha="center", va="center", fontsize=9)
        ax.annotate(
            "",
            xy=(center_x, center_y),
            xytext=(node_x, node_y),
            arrowprops=dict(
                arrowstyle="->",
                lw=line_width,
                color=color,
                alpha=0.85
            )
        )

        ax.text(
            (center_x + node_x) / 2,
            (center_y + node_y) / 2,
            f"coef={row['coef']:.3f}\ncorr={row['weighted_corr']:.3f}",
            fontsize=8,
            ha="center",
            va="center",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.8)
        )

    ax.set_title(
        f"Ego Network: {y_feature}\n{variable1} = {variable1_value}",
        fontsize=13
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def build_valid_x_summaries(
    df_coef: pd.DataFrame,
    pvalue_threshold: float = NETWORK_PVALUE_THRESHOLD
):
    group_cols = ["group_dim", "group_key", "y_feature"]

    corr_summary = (
        df_coef.sort_values(group_cols + ["x_feature"])
        .groupby(group_cols)
        .agg(
            corr_valid_x_count=("x_feature", "nunique"),
            corr_valid_x_features=("x_feature", lambda s: ", ".join(sorted(set(s))))
        )
        .reset_index()
    )

    df_reg_sig = df_coef[df_coef["p_value"] < pvalue_threshold].copy()
    if df_reg_sig.empty:
        reg_summary = pd.DataFrame(columns=[
            "group_dim", "group_key", "y_feature",
            "reg_valid_x_count", "reg_valid_x_features"
        ])
    else:
        reg_summary = (
            df_reg_sig.sort_values(group_cols + ["x_feature"])
            .groupby(group_cols)
            .agg(
                reg_valid_x_count=("x_feature", "nunique"),
                reg_valid_x_features=("x_feature", lambda s: ", ".join(sorted(set(s))))
            )
            .reset_index()
        )

    combined_summary = corr_summary.merge(
        reg_summary,
        on=group_cols,
        how="left"
    )
    combined_summary["reg_valid_x_count"] = (
        combined_summary["reg_valid_x_count"].fillna(0).astype(int)
    )
    combined_summary["reg_valid_x_features"] = (
        combined_summary["reg_valid_x_features"].fillna("")
    )

    return (
        corr_summary.sort_values(group_cols).reset_index(drop=True),
        reg_summary.sort_values(group_cols).reset_index(drop=True),
        combined_summary.sort_values(group_cols).reset_index(drop=True)
    )

# COMMAND ----------

FEATURE_POOL = [
    "07_01 채널_컨텐츠 APP", "07_02 구동성_구동속도_(1)TV 전반",
    "07_02 구동성_구동속도_(2)APP_웹전반", "07_03 메뉴 구성_UI",
    "07_04 SW_OS", "07_05 컨텐츠 탐색 사용성",
    "07_06 리모컨 사용성", "07_07 리모컨 디자인",
    "07_08 음성 인식_조작", "07_09 게임 기능",
    "07_10 부가 기능", "07_11 홈 IoT",
    "07_12 모바일 연동", "07_13 광고",
    "07_20 전반적 스마트 사용성"
]

FEATURE_POOL_CLEAN = clean_feature_pool(FEATURE_POOL)

model_rows = []
coef_rows = []
data_created_dt = pd.Timestamp(datetime.now())

for variable1 in ["all", "brand_name", "country", "year", "d_type"]:

    if variable1 == "all":
        group_keys = ["all"]
    else:
        group_keys = df_meta[variable1].dropna().unique()

    for key in group_keys:

        if variable1 == "all":
            pdf = df_wide.copy()
        else:
            model_ids = df_meta[df_meta[variable1] == key]["model_id"].unique()
            pdf = df_wide.loc[df_wide.index.isin(model_ids)]

        if pdf.empty:
            continue

        for y_feature in FEATURE_POOL_CLEAN:
            model, corr_info, x_obs_map = run_wls_weighted(
                pdf, y_feature, FEATURE_POOL_CLEAN
            )

            if model is None:
                continue

            model_rows.append({
                "y_feature": y_feature,
                "y_obs": int(model.nobs),
                "r_squared": safe_scalar(model.rsquared),
                "adj_r_squared": safe_scalar(model.rsquared_adj),
                "f_statistic": safe_scalar(model.fvalue),
                "prob_f": safe_scalar(model.f_pvalue),
                "log_likelihood": safe_scalar(model.llf),
                "aic": safe_scalar(model.aic),
                "bic": safe_scalar(model.bic),
                "cond_no": safe_scalar(model.condition_number),
                "group_dim": variable1,
                "group_key": str(key),
                "data_created_dt": data_created_dt,
            })

            for x, wcorr in corr_info:
                coef_rows.append({
                    "y_feature": y_feature,
                    "x_feature": x,
                    "coef": safe_scalar(model.params.get(f"{x}_score")),
                    "p_value": safe_scalar(model.pvalues.get(f"{x}_score")),
                    "t_value": safe_scalar(model.tvalues.get(f"{x}_score")),
                    "x_obs": x_obs_map.get(x),
                    "group_dim": variable1,
                    "group_key": str(key),
                    "weighted_corr": safe_scalar(wcorr),
                    "data_created_dt": data_created_dt,
                })

# COMMAND ----------

# DBTITLE 1,결과 스키마 정리 및 저장
df_model = pd.DataFrame(model_rows)
df_coef = pd.DataFrame(coef_rows)

if not df_coef.empty:
    df_coef["abs_coef"] = df_coef["coef"].abs()
    df_coef = df_coef.sort_values(
        ["group_dim", "group_key", "y_feature", "abs_coef"],
        ascending=[True, True, True, False]
    ).reset_index(drop=True)
    df_coef["driver_rank"] = (
        df_coef.groupby(["group_dim", "group_key", "y_feature"])["abs_coef"]
        .rank(method="first", ascending=False)
        .astype(int)
    )
    df_coef["is_driver"] = (
        (df_coef["p_value"] < DRIVER_PVALUE_THRESHOLD) &
        (df_coef["abs_coef"] >= DRIVER_COEF_THRESHOLD)
    ).astype(int)
    df_coef["variable1"] = df_coef["group_dim"]
    df_coef["variable1_value"] = df_coef["group_key"]
    df_coef = df_coef[
        [
            "variable1", "variable1_value",
            "y_feature", "x_feature", "coef", "p_value", "t_value",
            "x_obs", "group_dim", "group_key", "abs_coef",
            "driver_rank", "is_driver", "data_created_dt", "weighted_corr"
        ]
    ]

if not df_model.empty:
    df_model["variable1"] = df_model["group_dim"]
    df_model["variable1_value"] = df_model["group_key"]
    df_model["r2"] = df_model["r_squared"]
    df_model["n_obs"] = df_model["y_obs"]
    df_model = df_model[
        [
            "variable1", "variable1_value", "r2", "n_obs",
            "y_feature", "y_obs", "r_squared", "adj_r_squared",
            "f_statistic", "prob_f", "log_likelihood", "aic", "bic",
            "cond_no", "group_dim", "group_key", "data_created_dt"
        ]
    ]


In [0]:

spark.createDataFrame(df_model).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(MODEL_TABLE)
spark.createDataFrame(df_coef).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(COEF_TABLE)

print(f"모델 요약: {len(df_model)}건 저장 완료")
print(f"계수 상세: {len(df_coef)}건 저장 완료")

# COMMAND ----------

# DBTITLE 1,네트워크 edge 저장
df_network_edges = build_network_tables(
    df_model=df_model.drop(columns=["data_created_dt"]),
    df_coef=df_coef
)

if not df_network_edges.empty:
    spark.createDataFrame(df_network_edges).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        NETWORK_EDGE_TABLE
    )
    print(f"네트워크 edge: {len(df_network_edges)}건 저장 완료")
else:
    print("네트워크 edge 결과가 없어 저장을 생략했습니다.")

# COMMAND ----------

# DBTITLE 1,유의한 계수 필터링 (p < 0.05)
# 2) 유의한 계수 필터링 (p < 0.05)
df_sig = df_coef[df_coef["p_value"] < 0.05].sort_values(
    ["group_dim", "y_feature", "p_value"]
).reset_index(drop=True)

print(f"유의한 계수: {len(df_sig)}건 / 전체 {len(df_coef)}건")
display(spark.createDataFrame(df_sig))

# COMMAND ----------

# DBTITLE 1,R² 분포 시각화
# 3) R² 분포 시각화

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 전체 R² 히스토그램
axes[0].hist(df_model["r_squared"], bins=30, edgecolor="black", alpha=0.7)
axes[0].set_title("R² Distribution (All Groups)", fontsize=13)
axes[0].set_xlabel("R²")
axes[0].set_ylabel("Count")
axes[0].axvline(df_model["r_squared"].median(), color="red", linestyle="--", label=f'Median={df_model["r_squared"].median():.3f}')
axes[0].legend()

# 그룹별 R² 박스플롯
groups = df_model.groupby("group_dim")["r_squared"].apply(list)
axes[1].boxplot(groups.values, labels=groups.index)
axes[1].set_title("R² by Grouping Variable", fontsize=13)
axes[1].set_ylabel("R²")
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

# COMMAND ----------

# DBTITLE 1,선택형 Ego Network 및 그룹 인사이트
SELECTED_Y_FEATURE = "07_20_전반적_스마트_사용성"
SELECTED_GROUP_DIM = "country"
SELECTED_GROUP_KEY = "Korea"
TOP_N_EDGES = 8

selected_group_edges = df_network_edges[
    (df_network_edges["group_dim"] == SELECTED_GROUP_DIM) &
    (df_network_edges["group_key"] == SELECTED_GROUP_KEY) &
    (df_network_edges["y_feature"] == SELECTED_Y_FEATURE) &
    df_network_edges["is_significant"] &
    (df_network_edges["y_obs"] >= NETWORK_MIN_OBS)
].sort_values("abs_coef", ascending=False)

print(
    f"[선택 조건] y_feature={SELECTED_Y_FEATURE}, "
    f"group_dim={SELECTED_GROUP_DIM}, group_key={SELECTED_GROUP_KEY}"
)

if not selected_group_edges.empty:
    row = selected_group_edges.iloc[0]
    print(
        "네트워크 요약: "
        f"edge_count={len(selected_group_edges)}, "
        f"positive_edge_count={int((selected_group_edges['coef_sign'] == 'positive').sum())}, "
        f"negative_edge_count={int((selected_group_edges['coef_sign'] == 'negative').sum())}, "
        f"top_driver={row['x_feature']}"
    )
else:
    print("선택 조건에 해당하는 네트워크 edge가 없습니다.")

print("선택한 y_feature 기준 유의한 incoming edge")
display(
    spark.createDataFrame(
        selected_group_edges.head(20)
        if not selected_group_edges.empty else
        pd.DataFrame(columns=df_network_edges.columns)
    )
)

plot_ego_network(
    df_edges=df_network_edges,
    y_feature=SELECTED_Y_FEATURE,
    variable1=SELECTED_GROUP_DIM,
    variable1_value=SELECTED_GROUP_KEY,
    top_n=TOP_N_EDGES
)
